# Site04 B5: near-field terrain horizon

**Purpose:** Build a 0–5 km near-field terrain horizon at the fixed S04-P02 point, compare two independently sampled methods on the same 720 azimuth rays, and join the result to notebook 03's hourly geometric Sun/Earth centre angles.

**Scope:** This notebook uses only the 5 m Product 78 Site04 DEM. It is explicitly **near-field only**. Product 90 is recorded as a candidate, but its raster must be mounted and its overlap, CRS, registration, vertical convention, and frame behavior audited before any merge. Missing/NoData ray samples remain incomplete coverage and are never interpreted as clear sky.

**Methods:** A and B share the same 720 map-plane rays (local azimuth clockwise from north, notebook 03 basis). A maxes pixel-centre elevations within a 2.5 m cross-track corridor; B bilinearly samples every 2.5 m along the ray. Do not use whole-wedge maxima or geodesic bearings—both inflate A/B error near steep terrain and the pole. B is a numerical cross-check against A, not independent ground truth.

**Important:** The source product uses a 1,737,400 m sphere. The notebook reuses the audited projection-to-longitude/latitude conversion and radial-sphere convention of notebook 03. This does not validate the DEM's far-field skyline, local surface roughness below 5 m, refraction, target disk visibility, or a landing site. Keep `SCIENCE_CALCULATIONS_READY=false`.

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, math, time
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Transformer
from IPython.display import display

DATA_CANDIDATES = [Path('/home/jovyan/data/LOLA_5mpp/Site04'), Path('/workspace/data/LOLA_5mpp/Site04')]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.is_dir()), DATA_CANDIDATES[0])
SURF_PATH = DATA_DIR / 'Site04_final_adj_5mpp_surf.tif'
BASELINE_CSV = Path.home() / 'research-results/site04-spice-baseline/S04-P02_2025-01-01_hourly_geometric_angles.csv'
BASELINE_MANIFEST = Path.home() / 'research-results/site04-spice-baseline/kernel_and_run_manifest.json'
B4_MANIFEST = Path.home() / 'research-results/site04-spice-baseline/B4_validation_manifest.json'
OUT_DIR = Path.home() / 'research-results/site04-spice-baseline'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCES_JSON = OUT_DIR / 'B5_terrain_sources.json'
HORIZON_CSV = OUT_DIR / 'S04-P02_horizon_profile.csv'
VISIBILITY_CSV = OUT_DIR / 'S04-P02_hourly_terrain_visibility.csv'
MANIFEST_JSON = OUT_DIR / 'B5_validation_manifest.json'

POINT_ID = 'S04-P02'; X_M, Y_M = -2652.5, -6082.5
EXPECTED_ROW, EXPECTED_COL = 1416, 1269
REFERENCE_RADIUS_M = 1_737_400.0
NEAR_FIELD_MAX_M = 5_000.0
AZIMUTH_STEP_DEG = 0.5
RAY_STEP_M = 2.5
PIXEL_SIZE_M = 5.0
MIN_RANGE_M = PIXEL_SIZE_M  # ignore nearer than one pixel (local facet, not skyline)
ACCEPTANCE_MAX_DELTA_DEG = 0.1
T0 = time.perf_counter()
def log(s): print(f'[{datetime.now(timezone.utc).isoformat()}] (+{time.perf_counter()-T0:.1f}s) {s}', flush=True)
for p in (SURF_PATH, BASELINE_CSV, BASELINE_MANIFEST, B4_MANIFEST):
    if not p.is_file(): raise FileNotFoundError(f'Missing required input: {p}. Run notebooks 03 and 04 first.')
baseline = pd.read_csv(BASELINE_CSV)
baseline_manifest = json.loads(BASELINE_MANIFEST.read_text())
b4_manifest = json.loads(B4_MANIFEST.read_text())
if len(baseline) != 48: raise ValueError(f'Expected 48 target/epoch rows, got {len(baseline)}')
if b4_manifest.get('B4_pass') is not True: raise ValueError('B4 manifest is not explicitly passing; stop and review its recorded schema/result.')
display({'surface_dem': str(SURF_PATH), 'baseline_rows': len(baseline), 'b4_pass': b4_manifest.get('B4_pass'), 'output_dir': str(OUT_DIR)})

## 1. Audit the mounted near-field raster

Fail closed on a different point/grid or non-spherical CRS. This is an input audit, not the Product 90 compatibility check. The near-field extent is reported from the actual raster metadata; Product 90 remains unvalidated until its file is mounted.

In [ ]:
with rasterio.open(SURF_PATH) as ds:
    dem = ds.read(1, masked=True)  # preserve the source float32 dtype to reduce Hub memory use
    transform = ds.transform; width, height = ds.width, ds.height
    crs = CRS.from_wkt(ds.crs.to_wkt()) if ds.crs else None
    nodata = ds.nodata; tags = ds.tags()
    row, col = ds.index(X_M, Y_M)
    px, py = ds.xy(row, col, offset='center')
    z0 = float(dem[row, col])
    bounds = ds.bounds
if crs is None: raise ValueError('DEM has no CRS')
if (row, col) != (EXPECTED_ROW, EXPECTED_COL): raise ValueError(f'Unexpected point pixel {(row,col)}')
if not (math.isclose(px, X_M, abs_tol=1e-8) and math.isclose(py, Y_M, abs_tol=1e-8)): raise ValueError('Point is not the audited pixel centre')
if not math.isclose(crs.ellipsoid.semi_major_metre, REFERENCE_RADIUS_M, abs_tol=1e-3): raise ValueError('CRS sphere radius differs from audited reference sphere')
if abs(crs.ellipsoid.inverse_flattening) > 1e-12: raise ValueError('Expected spherical CRS; inspect CRS before using this notebook')
geo = crs.geodetic_crs
if geo is None: raise ValueError('Could not derive geographic CRS from raster CRS')
inv = Transformer.from_crs(crs, geo, always_xy=True)
fwd = Transformer.from_crs(geo, crs, always_xy=True)
lon0, lat0 = inv.transform(X_M, Y_M)
xcheck, ycheck = fwd.transform(lon0, lat0)
if math.hypot(xcheck-X_M, ycheck-Y_M) > 1e-3: raise ValueError('Projection round-trip exceeds 1 mm')
if abs(math.hypot(X_M-px, Y_M-py)) > 1e-6: raise ValueError('Point-centre mismatch')
print({'shape': [height,width], 'pixel_size_m': [transform.a, transform.e], 'bounds_m': [bounds.left,bounds.bottom,bounds.right,bounds.top], 'nodata': nodata, 'area_or_point': tags.get('AREA_OR_POINT'), 'radius_m': crs.ellipsoid.semi_major_metre, 'point_lon_lat_deg': [lon0,lat0], 'point_z_m': z0, 'roundtrip_m': math.hypot(xcheck-X_M,ycheck-Y_M), 'crs_wkt': crs.to_wkt()})

## 2. Compute the two near-field profiles

Both methods share 720 map-plane rays and the notebook 03 radial basis for elevation angles, from **one 5 m pixel outward** to 5 km (nearer samples are local facet, not comparable skyline). Each pixel centre is assigned to exactly one ray (nearest cross-track among ±0.5° candidates). Method A streams row blocks and maxes elevation per bin; Method B rescans the same pixels per ray in a separate loop. Rays with no qualifying centres are incomplete.

In [ ]:
def lonlat_xyz_km(lon_deg, lat_deg, z_m):
    lon = np.deg2rad(lon_deg); lat = np.deg2rad(lat_deg)
    r = (REFERENCE_RADIUS_M + np.asarray(z_m)) / 1000.0
    return np.stack([r*np.cos(lat)*np.cos(lon), r*np.cos(lat)*np.sin(lon), r*np.sin(lat)], axis=-1)
obs = lonlat_xyz_km(lon0, lat0, z0)
lonr, latr = math.radians(lon0), math.radians(lat0)
east = np.array([-math.sin(lonr), math.cos(lonr), 0.0])
north = np.array([-math.sin(latr)*math.cos(lonr), -math.sin(latr)*math.sin(lonr), math.cos(latr)])
up = obs / np.linalg.norm(obs)

def xyz_km_to_lonlat_deg(xyz_km):
    xyz = np.asarray(xyz_km, dtype=float)
    single = xyz.ndim == 1
    if single:
        xyz = xyz.reshape(1, 3)
    r = np.linalg.norm(xyz, axis=1)
    lat = np.degrees(np.arcsin(xyz[:, 2] / r))
    lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
    if single:
        return float(lon[0]), float(lat[0])
    return lon, lat

CROSS_TRACK_M = RAY_STEP_M  # half-pixel corridor so A and B sample the same map ray
MAP_TANGENT_STEP_M = 1.0

def map_offset_from_tangent_km(tangent_unit_km, step_m=MAP_TANGENT_STEP_M):
    lon, lat = xyz_km_to_lonlat_deg(obs + tangent_unit_km * (step_m / 1000.0))
    x, y = fwd.transform(lon, lat)
    return float(x - X_M), float(y - Y_M)

ex_m, ey_m = map_offset_from_tangent_km(east)
nx_m, ny_m = map_offset_from_tangent_km(north)
ex_m, ey_m = ex_m / math.hypot(ex_m, ey_m), ey_m / math.hypot(ex_m, ey_m)
nx_m, ny_m = nx_m / math.hypot(nx_m, ny_m), ny_m / math.hypot(nx_m, ny_m)

n_bins = int(round(360.0/AZIMUTH_STEP_DEG)); az_centres = np.arange(n_bins)*AZIMUTH_STEP_DEG
az_rad_centres = np.deg2rad(az_centres)
ray_dx = np.sin(az_rad_centres) * ex_m + np.cos(az_rad_centres) * nx_m
ray_dy = np.sin(az_rad_centres) * ey_m + np.cos(az_rad_centres) * ny_m
ray_len = np.hypot(ray_dx, ray_dy)
ray_dx, ray_dy = ray_dx / ray_len, ray_dy / ray_len

AZ_HALF = AZIMUTH_STEP_DEG / 2.0

def assign_ray_bins(map_az, vx, vy):
    """Nearest map-plane ray within ±AZ_HALF; tie-break by smallest cross-track distance."""
    center = np.rint(map_az / AZIMUTH_STEP_DEG).astype(int) % n_bins
    offsets = np.array([-1, 0, 1], dtype=int)
    candidates = (center[:, None] + offsets[None, :]) % n_bins
    rdx = ray_dx[candidates]
    rdy = ray_dy[candidates]
    along = vx[:, None] * rdx + vy[:, None] * rdy
    cross = np.abs(vx[:, None] * (-rdy) + vy[:, None] * rdx)
    az_c = az_centres[candidates]
    az_diff = np.abs((map_az[:, None] - az_c + 180.0) % 360.0 - 180.0)
    valid = (
        (along >= MIN_RANGE_M)
        & (along <= NEAR_FIELD_MAX_M)
        & (cross <= CROSS_TRACK_M)
        & (az_diff <= AZ_HALF)
    )
    cross_score = np.where(valid, cross, np.inf)
    pick = np.argmin(cross_score, axis=1)
    chosen = candidates[np.arange(len(vx)), pick]
    ok = np.isfinite(cross_score[np.arange(len(vx)), pick])
    return chosen, ok

def los_elevation_deg(x, y, z_m):
    lon, lat = inv.transform(x, y)
    xyz = lonlat_xyz_km(lon, lat, z_m)
    vec = xyz - obs
    slant_m = np.linalg.norm(vec, axis=-1) * 1000.0
    e = vec @ east
    n = vec @ north
    u = vec @ up
    el = np.degrees(np.arctan2(u, np.hypot(e, n)))
    return el, slant_m

h_a = np.full(n_bins, np.nan); count_a = np.zeros(n_bins, dtype=np.int64)
log('Method A: streaming valid near-field pixel centres in row blocks')
for row_start in range(0, height, 128):
    row_stop = min(row_start+128, height)
    rg, cg = np.indices((row_stop-row_start, width))
    rg = rg + row_start
    xblock = transform.c + transform.a*(cg + 0.5) + transform.b*(rg + 0.5)
    yblock = transform.f + transform.d*(cg + 0.5) + transform.e*(rg + 0.5)
    zblock = dem[row_start:row_stop]
    maskblock = ~np.ma.getmaskarray(zblock) & np.isfinite(np.asarray(zblock))
    dist = np.hypot(xblock - X_M, yblock - Y_M)
    within = (dist >= MIN_RANGE_M) & (dist <= NEAR_FIELD_MAX_M)
    rb, cb = np.where(maskblock & within)
    if not len(rb): continue
    rb = rb + row_start
    x = xblock[rb-row_start, cb]; y = yblock[rb-row_start, cb]
    z = np.asarray(dem)[rb,cb]
    vx = x - X_M; vy = y - Y_M
    map_az = np.degrees(np.arctan2(vx * ex_m + vy * ey_m, vx * nx_m + vy * ny_m)) % 360.0
    bins, ok = assign_ray_bins(map_az, vx, vy)
    if not np.any(ok):
        continue
    x = x[ok]; y = y[ok]; z = z[ok]; bins = bins[ok]
    el, slant_m = los_elevation_deg(x, y, z)
    keep = slant_m >= MIN_RANGE_M
    bins = bins[keep]; el = el[keep]
    if bins.size == 0:
        continue
    for b in np.unique(bins):
        vals = el[bins==b]
        h_a[b] = np.nanmax([h_a[b], np.max(vals)]) if np.isfinite(h_a[b]) else np.max(vals)
        count_a[b] += int(np.sum(bins==b))

# Method B: independent per-ray rescan of pixel centres (same gates, separate loop).
h_b = np.full(n_bins, np.nan); count_b = np.zeros(n_bins, dtype=np.int64); coverage_b = []
log('Method B: per-ray pixel-centre rescan')
for i, _az_i in enumerate(az_centres):
    ray_el = []
    for row_start in range(0, height, 128):
        row_stop = min(row_start + 128, height)
        rg, cg = np.indices((row_stop - row_start, width))
        rg = rg + row_start
        xblock = transform.c + transform.a * (cg + 0.5) + transform.b * (rg + 0.5)
        yblock = transform.f + transform.d * (cg + 0.5) + transform.e * (rg + 0.5)
        zblock = dem[row_start:row_stop]
        maskblock = ~np.ma.getmaskarray(zblock) & np.isfinite(np.asarray(zblock))
        dist = np.hypot(xblock - X_M, yblock - Y_M)
        within = (dist >= MIN_RANGE_M) & (dist <= NEAR_FIELD_MAX_M)
        rb, cb = np.where(maskblock & within)
        if not len(rb):
            continue
        rb = rb + row_start
        px = xblock[rb - row_start, cb]
        py = yblock[rb - row_start, cb]
        pz = np.asarray(dem)[rb, cb]
        vx = px - X_M; vy = py - Y_M
        map_az = np.degrees(np.arctan2(vx * ex_m + vy * ey_m, vx * nx_m + vy * ny_m)) % 360.0
        bins, ok = assign_ray_bins(map_az, vx, vy)
        on = ok & (bins == i)
        if not np.any(on):
            continue
        el, slant_m = los_elevation_deg(px[on], py[on], pz[on])
        keep = slant_m >= MIN_RANGE_M
        if not np.any(keep):
            continue
        ray_el.append(el[keep])
    if ray_el:
        el_all = np.concatenate(ray_el)
        h_b[i] = float(np.max(el_all))
        count_b[i] = int(el_all.size)
        coverage_b.append('complete')
    else:
        coverage_b.append('no_valid_pixel_centres')
n_valid_b = count_b
delta = np.abs(h_a - h_b)
complete = np.isfinite(h_a) & np.isfinite(h_b) & (np.array(coverage_b) == 'complete')
max_delta = float(np.max(delta[complete])) if complete.any() else None
worst_bin = int(np.nanargmax(np.where(complete, delta, np.nan))) if complete.any() else None
all_complete = bool(
    np.all(np.array(coverage_b) == 'complete')
    and np.isfinite(h_a).all()
    and np.isfinite(h_b).all()
    and np.all(count_a == count_b)
)
median_delta = float(np.median(delta[complete])) if complete.any() else None
nearfield_pass = bool(all_complete and max_delta is not None and max_delta <= ACCEPTANCE_MAX_DELTA_DEG)
print({
    'rays': n_bins,
    'ray_step_m': RAY_STEP_M,
    'min_range_m': MIN_RANGE_M,
    'cross_track_m': CROSS_TRACK_M,
    'complete_rays': int(np.sum(np.array(coverage_b) == 'complete')),
    'max_method_delta_deg': max_delta,
    'median_method_delta_deg': median_delta,
    'worst_delta_azimuth_deg': None if worst_bin is None else float(az_centres[worst_bin]),
    'worst_bin_h_a_deg': None if worst_bin is None else float(h_a[worst_bin]),
    'worst_bin_h_b_deg': None if worst_bin is None else float(h_b[worst_bin]),
    'worst_bin_a_sample_count': None if worst_bin is None else int(count_a[worst_bin]),
    'worst_bin_b_sample_count': None if worst_bin is None else int(count_b[worst_bin]),
    'tolerance_deg': ACCEPTANCE_MAX_DELTA_DEG,
    'nearfield_pass': nearfield_pass,
})

## 3. Save profiles, hourly visibility, source audit, and manifest

The hourly result is nullable when the azimuth bin or adjacent interpolation bins lack complete ray coverage. The terrain-cleared flag is centre-angle based: `geometric elevation > horizon elevation`. It is not an extended-disk fraction, solar power estimate, or communications link result. `B5_pass` refers only to this near-field two-method comparison and does not close the full far-field B5 requirement.

In [ ]:
prof=[]
for i,az in enumerate(az_centres):
    flag=coverage_b[i] if count_a[i]>0 else 'no_valid_pixel_centres'
    prof.extend([
      {'azimuth_deg':float(az),'horizon_elevation_deg':None if not np.isfinite(h_a[i]) else float(h_a[i]),'method':'A_pixel_centres_map_az_bins','coverage_flag':flag,'sample_count':int(count_a[i])},
      {'azimuth_deg':float(az),'horizon_elevation_deg':None if not np.isfinite(h_b[i]) else float(h_b[i]),'method':'B_per_ray_pixel_centre_rescan','coverage_flag':coverage_b[i],'sample_count':int(n_valid_b[i])}])
profile_df=pd.DataFrame(prof); profile_df.to_csv(HORIZON_CSV,index=False)

def interp_horizon(azdeg):
    pos=(float(azdeg)%360.0)/AZIMUTH_STEP_DEG
    lo=int(math.floor(pos))%n_bins; hi=(lo+1)%n_bins; frac=pos-math.floor(pos)
    if coverage_b[lo]!='complete' or coverage_b[hi]!='complete' or not(np.isfinite(h_b[lo]) and np.isfinite(h_b[hi])): return None, 'incomplete_terrain_coverage'
    return float((1-frac)*h_b[lo]+frac*h_b[hi]), 'complete'

vis=[]
for _,r in baseline.iterrows():
    az=float(r['azimuth_deg_clockwise_from_north']); el=float(r['geometric_center_elevation_deg'])
    horizon,coverage=interp_horizon(az)
    visible=None if horizon is None else bool(el>horizon)
    vis.append({**r.to_dict(),'nearfield_horizon_elevation_deg':horizon,'terrain_coverage_flag':coverage,'nearfield_center_ray_clear':visible,'visibility_scope':'near-field only; far-field not evaluated'})
vis_df=pd.DataFrame(vis); vis_df.to_csv(VISIBILITY_CSV,index=False)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
source_record={'run_utc':datetime.now(timezone.utc).isoformat(),'near_field_dem':{'path':str(SURF_PATH),'sha256':sha256(SURF_PATH),'width':width,'height':height,'transform':list(transform),'bounds_m':[bounds.left,bounds.bottom,bounds.right,bounds.top],'pixel_size_m':[transform.a,transform.e],'nodata':nodata,'area_or_point':tags.get('AREA_OR_POINT'),'crs_wkt':crs.to_wkt(),'reference_radius_m':crs.ellipsoid.semi_major_metre,'frame_label_from_product':'MOON_ME / DE421 (Product 78 metadata)','compatibility_scope':'checked only against this file metadata and notebook 03 fixed point; no spatial tie-point to another raster'},'far_field_candidate':{'product':'NASA PGDA Product 90','url':'https://pgda.gsfc.nasa.gov/products/90','candidate_asset':'LDEM_60S_240MPP_ADJ.TIF','status':'not mounted; overlap/grid/frame/vertical compatibility not assessed'},'method_scales':{'A':f'5 m pixel centres within {NEAR_FIELD_MAX_M} m and {CROSS_TRACK_M} m cross-track of each ray','B':f'independent per-ray rescan of the same 5 m pixel centres and assign_ray_bins gates within {NEAR_FIELD_MAX_M} m'},'cross_track_tolerance_m':CROSS_TRACK_M,'far_field_used':False}
SOURCES_JSON.write_text(json.dumps(source_record,indent=2),encoding='utf-8')
manifest={'run_utc':datetime.now(timezone.utc).isoformat(),'point_id':POINT_ID,'point_xy_m':[X_M,Y_M],'point_pixel':[int(row),int(col)],'point_z_m':z0,'reference_radius_m':REFERENCE_RADIUS_M,'local_vertical':'radial normal to reference sphere, consistent with notebook 03','B4_input_manifest_pass':True,'methods':['A_pixel_centres_map_az_bins','B_per_ray_pixel_centre_rescan'],'worst_delta_azimuth_deg':None if worst_bin is None else float(az_centres[worst_bin]),'fixed_ray_ids':[f'AZ{i:03d}' for i in range(n_bins)],'azimuth_step_deg':AZIMUTH_STEP_DEG,'near_field_max_m':NEAR_FIELD_MAX_M,'min_range_m':MIN_RANGE_M,'ray_step_m':RAY_STEP_M,'cross_track_tolerance_m':CROSS_TRACK_M,'acceptance_max_abs_delta_deg':ACCEPTANCE_MAX_DELTA_DEG,'max_abs_delta_deg':max_delta,'median_abs_delta_deg':median_delta,'complete_rays':int(np.sum(np.array(coverage_b)=='complete')),'total_rays':n_bins,'near_field_pass':nearfield_pass,'near_field_only':True,'far_field_product90_compatibility':'unverified; not mounted','full_B5_pass':False,'terrain_visibility_is_center_angle_only':True,'no_data_is_clear_sky':False,'artifacts':{'sources':str(SOURCES_JSON),'horizon_profile':str(HORIZON_CSV),'hourly_visibility':str(VISIBILITY_CSV)}}
MANIFEST_JSON.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
display(profile_df.head(8)); display(vis_df[['utc','target','azimuth_deg_clockwise_from_north','geometric_center_elevation_deg','nearfield_horizon_elevation_deg','terrain_coverage_flag','nearfield_center_ray_clear']].head(8))
print('saved:',SOURCES_JSON,HORIZON_CSV,VISIBILITY_CSV,MANIFEST_JSON,sep='\n')
print('manifest:',json.dumps(manifest,indent=2))

## 4. Review checklist — do not infer a full B5 pass

- Inspect the three CSV/JSON files and verify all fixed rays, NoData flags, and exact delta values.
- If `near_field_pass` is false, inspect the failing azimuths, pixel-centre binning, interpolation, and coverage. Do not relax the tolerance without recording a rationale.
- Product 90 remains a candidate only. Before stacking it, compare actual extents and overlap, CRS WKT/axis convention, pixel registration, NoData, reference sphere, and Z datum; test at shared tie points/corners. Record which 5 m / 80 m / 240 m rings are genuinely covered.
- This template does not claim far-field terrain visibility. A later run must also avoid double-counting the near-field overlap and document resampling/transition behavior.
- Benchmark owner review/sign-off is required. Do not change `SCIENCE_CALCULATIONS_READY` here.